# VLM 智能图文问答助手 - Colab 快速演示
---
**用途**: 在 Google Colab T4 上快速体验 Qwen2.5-VL 7B 图文问答

**使用**:
1. 将 project.zip 上传到你的 Google Drive
2. 在 Colab 中打开此 Notebook
3. 依次运行各 Cell

In [ ]:
# ============================================================
# Cell 1: 挂载 Google Drive & 环境准备
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate bitsandbytes qwen-vl-utils datasets pillow pyyaml tqdm gradio

import os
import sys
import zipfile

# 解压项目
zip_path = "/content/drive/MyDrive/project.zip"  # 根据实际路径修改
project_path = "/content/vlm_project"

if os.path.exists(zip_path):
    os.makedirs(project_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(project_path)
    print(f"[OK] 项目已解压到 {project_path}")
else:
    print(f"[!] 未找到 {zip_path}，请确保已将 project.zip 上传到 Google Drive")
    # 备用方案：直接 clone
    # !git clone <你的仓库> {project_path}

sys.path.insert(0, project_path)
os.chdir(project_path)

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ============================================================
# Cell 2: 加载模型（4-bit 量化）
# ============================================================

from src.inference.model_loader import load_model_and_processor

model, processor = load_model_and_processor(
    model_name="Qwen/Qwen2.5-VL-7B-Instruct",
    use_quantization=True,
)

print("[OK] 模型加载完成！")

In [ ]:
# ============================================================
# Cell 3: 单图问答测试
# ============================================================

from src.inference.engine import ask
from src.preprocess.resize_adapter import resize_to_vl
from src.prompt.builder import build_baseline_prompt
from PIL import Image
import requests
from io import BytesIO

# 下载一张示例图片
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
resp = requests.get(url)
image = Image.open(BytesIO(resp.content)).convert("RGB")
image = resize_to_vl(image)

# 提问
question = "What is in this image?"
prompt = build_baseline_prompt(question)

print(f"问题: {question}")
print(f"Prompt: {prompt}")

answer = ask(image, prompt)
print(f"\n回答: {answer}")

# 显示图片
display(image)

In [ ]:
# ============================================================
# Cell 4: 多问题测试
# ============================================================

from src.prompt.builder import build_prompt

questions = [
    ("What color is the car?", "general"),
    ("How many wheels can you see?", "general"),
    ("图中有什么物体？请用中文回答", "chinese"),
]

for q, tpl in questions:
    prompt = build_prompt(q, template_name=tpl)
    ans = ask(resize_to_vl(image), prompt)
    print(f"[{tpl}] 问: {q}")
    print(f"      答: {ans}")
    print()

In [ ]:
# ============================================================
# Cell 5 (可选): 启动 Gradio 演示界面
# ============================================================
# 注意: Colab 免费版 Gradio 可能不稳定，建议用 ngrok 或直接运行 run_eval

# !pip install -q gradio
# !python app.py --share